# EyeAI AMD3 — Prepare HYAMD + ARMD Curated Binary Dataset

This notebook is intentionally separate from model training.

It performs:
- HYAMD preparation with patient-level splits.
- ARMD Curated positive-only audit.
- Exact and high-confidence near-duplicate removal.
- Outer black-border cropping only; no CLAHE or fixed gamma filter.
- A duplicate-group-safe ARMD split: 90% training and 10% positive-only external validation.
- Portable manifests for fixed-split and 3-fold experiments.
- Three random before/after preprocessing examples.

HYAMD validation remains the primary model-selection set. ARMD validation is reported separately because it contains positive images only.


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess

REPO_OWNER = "MozaicAI-Solutions"
REPO_NAME = "eyeai-team-AMD3"
BRANCH = "main"
REPO_DIR = Path("/kaggle/working/eyeai-team-AMD3")
PREP_CONFIG = "configs/prepare_hyamd_armd_binary.yaml"
PREPARED_DIR = Path("/kaggle/working/eyeai_prepared_binary_dataset")

print("Repository:", REPO_DIR)
print("Prepared dataset output:", PREPARED_DIR)


In [ ]:
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
github_token = secrets.get_secret("GITHUB_TOKEN")
if not github_token:
    raise RuntimeError("GITHUB_TOKEN secret is missing.")

repo_url = f"https://{github_token}@github.com/{REPO_OWNER}/{REPO_NAME}.git"
if REPO_DIR.exists() and (REPO_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "origin", BRANCH], check=True)
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(["git", "clone", "--branch", BRANCH, repo_url, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
print(subprocess.run(["git", "status", "--short"], capture_output=True, text=True, check=True).stdout)


In [ ]:
os.chdir(REPO_DIR)
subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run(["pip", "install", "-q", "-e", "."], check=True)
print("EyeAI package installed.")


In [ ]:
import yaml

print("Kaggle input directories:")
for path in sorted(Path("/kaggle/input").glob("*")):
    print("-", path)

config_path = REPO_DIR / PREP_CONFIG
with open(config_path, "r", encoding="utf-8") as handle:
    config = yaml.safe_load(handle)
print("\nPreparation config:")
print(yaml.safe_dump(config, sort_keys=False))


## Optional split reuse

To preserve an already approved HYAMD split, edit `existing_split_dir` in the preparation config before running the next cell. The directory must contain `train.csv`, `val.csv`, and `test.csv` with `image_id` columns.

Leave it as `null` to create a new deterministic patient-level split with seed 42.


In [ ]:
os.chdir(REPO_DIR)
subprocess.run(
    ["python", "-u", "scripts/prepare_binary_dataset.py", "--config", PREP_CONFIG],
    check=True,
)
print("Preparation completed.")


In [ ]:
from pathlib import Path
import random

import matplotlib.pyplot as plt
import pandas as pd
import yaml
from PIL import Image

# Reload paths so this cell is safe to run independently
with open(REPO_DIR / PREP_CONFIG, "r", encoding="utf-8") as handle:
    preparation_cfg = yaml.safe_load(handle)["preparation"]

prepared_dir = Path(preparation_cfg["output_dir"])
intermediate_dir = Path(preparation_cfg["intermediate_dir"])

# HYAMD raw-to-prepared mapping
hyamd_build_csv = intermediate_dir / "HYAMD_raw" / "final_hyamd_dataframe.csv"
hyamd_build = pd.read_csv(hyamd_build_csv)
hyamd_manifest = pd.read_csv(prepared_dir / "manifests" / "hyamd_all.csv")

hyamd_examples = hyamd_build.merge(
    hyamd_manifest[["image_id", "relative_image_path", "dataset_source"]],
    on="image_id",
    how="inner",
    validate="one_to_one",
)
hyamd_examples["before_path"] = hyamd_examples["image_path"].astype(str)
hyamd_examples["after_path"] = hyamd_examples["relative_image_path"].map(
    lambda value: str(prepared_dir / value)
)

# ARMD source-to-prepared mapping
image_audit = pd.read_csv(prepared_dir / "audit" / "image_audit.csv")
external_manifest = pd.read_csv(prepared_dir / "manifests" / "armd_curated_all_clean.csv")
external_audit = image_audit[
    image_audit["dataset_source"].astype(str).str.lower().eq("armd_curated")
][["sha256", "source_path"]]

external_examples = external_manifest.merge(
    external_audit,
    on="sha256",
    how="inner",
    validate="one_to_one",
)
external_examples["before_path"] = external_examples["source_path"].astype(str)
external_examples["after_path"] = external_examples["relative_image_path"].map(
    lambda value: str(prepared_dir / value)
)

# Select two HYAMD images and one external AMD image
selected = pd.concat(
    [
        hyamd_examples.sample(n=min(2, len(hyamd_examples)), random_state=42),
        external_examples.sample(n=min(1, len(external_examples)), random_state=42),
    ],
    ignore_index=True,
)

if len(selected) < 3:
    all_examples = pd.concat([hyamd_examples, external_examples], ignore_index=True)
    selected_ids = set(selected["image_id"].astype(str))
    remaining = all_examples[~all_examples["image_id"].astype(str).isin(selected_ids)]
    selected = pd.concat(
        [selected, remaining.sample(n=min(3 - len(selected), len(remaining)), random_state=43)],
        ignore_index=True,
    )

fig, axes = plt.subplots(len(selected), 2, figsize=(12, 4.5 * len(selected)))
if len(selected) == 1:
    axes = [axes]

for row_index, row in selected.iterrows():
    before_path = Path(row["before_path"])
    after_path = Path(row["after_path"])
    if not before_path.exists():
        raise FileNotFoundError(before_path)
    if not after_path.exists():
        raise FileNotFoundError(after_path)

    with Image.open(before_path) as opened:
        before_image = opened.convert("RGB")
        before_size = before_image.size
        axes[row_index][0].imshow(before_image)

    with Image.open(after_path) as opened:
        after_image = opened.convert("RGB")
        after_size = after_image.size
        axes[row_index][1].imshow(after_image)

    source = str(row.get("dataset_source", "unknown"))
    image_name = str(row.get("image_name", row.get("image_id", "image")))
    axes[row_index][0].set_title(f"Before — {source}\n{image_name} | {before_size}")
    axes[row_index][1].set_title(f"After — {source}\n{image_name} | {after_size}")
    axes[row_index][0].axis("off")
    axes[row_index][1].axis("off")

plt.suptitle("Fundus images before and after preprocessing", fontsize=16, y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
import json
import pandas as pd

summary_path = PREPARED_DIR / "dataset_summary.json"
if not summary_path.exists():
    raise FileNotFoundError(summary_path)

summary = json.loads(summary_path.read_text(encoding="utf-8"))
print(json.dumps(summary, indent=2))

manifest_paths = [
    "manifests/hyamd_train.csv",
    "manifests/hyamd_val.csv",
    "manifests/hyamd_test_locked.csv",
    "manifests/armd_curated_train.csv",
    "manifests/train_mixed.csv",
]
for relative_path in manifest_paths:
    path = PREPARED_DIR / relative_path
    frame = pd.read_csv(path)
    print("\n", relative_path, len(frame))
    print(frame.groupby(["dataset_source", "binary_label"]).size())


In [ ]:
audit_path = PREPARED_DIR / "audit" / "image_audit.csv"
exclusions_path = PREPARED_DIR / "audit" / "external_exclusions.csv"

audit = pd.read_csv(audit_path)
exclusions = pd.read_csv(exclusions_path)
print("Audit rows:", len(audit))
print(audit.groupby(["dataset_source", "status"]).size())
print("\nExternal exclusions:", len(exclusions))
if len(exclusions):
    print(exclusions["exclusion_reason"].value_counts())


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

hyamd = pd.read_csv(PREPARED_DIR / "manifests" / "hyamd_train.csv")
external = pd.read_csv(PREPARED_DIR / "manifests" / "armd_curated_train.csv")

samples = pd.concat([
    hyamd.sample(min(4, len(hyamd)), random_state=42),
    external.sample(min(4, len(external)), random_state=42),
], ignore_index=True)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for axis, row in zip(axes.flatten(), samples.itertuples(index=False)):
    image = Image.open(PREPARED_DIR / row.relative_image_path).convert("RGB")
    axis.imshow(image)
    axis.set_title(f"{row.dataset_source} | y={row.binary_label}")
    axis.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
required = [
    PREPARED_DIR / "dataset_summary.json",
    PREPARED_DIR / "manifests" / "train_mixed.csv",
    PREPARED_DIR / "manifests" / "armd_curated_val_positive.csv",
    PREPARED_DIR / "manifests" / "hyamd_test_locked.csv",
    PREPARED_DIR / "manifests" / "folds" / "fold_0_train_mixed.csv",
]
for path in required:
    print(path, "exists=", path.exists())
    if not path.exists():
        raise FileNotFoundError(path)

print("\nThe prepared directory is ready to be saved as a Kaggle Notebook output dataset:")
print(PREPARED_DIR)
print("Use Kaggle Save Version, then create a dataset from the notebook output.")
